# Tetris OpenEnv - Training with Unsloth + GRPO

Train an LLM agent to play Tetris using GRPO (Group Relative Policy Optimization).

**Environment**: Tetris on HF Spaces via OpenEnv 0.2.1
**Model**: Qwen2.5-7B-Instruct (4-bit via Unsloth)
**Training**: GRPO via TRL

Runtime: GPU T4 (free tier Colab)

In [ ]:
# Cell 1: Install dependencies
!pip install unsloth trl openenv-core datasets -q

In [ ]:
# Cell 2: Load model with Unsloth (4-bit quantization for T4)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

In [ ]:
# Cell 4: Generate training prompts from random board states (LOCAL, fast)
import random

# Download game engine to run locally (no network calls needed)
\!wget -q https://raw.githubusercontent.com/OutOfMystic/tetris-openenv/main/src/tetris_env/server/game_engine.py

from game_engine import TetrisEnv

SYSTEM_PROMPT = """You are a Tetris AI agent. You see a 10x20 board and must choose the best action.
Actions: left, right, rotate_cw, rotate_ccw, drop, down, noop
Your goal: clear as many lines as possible. Clearing multiple lines at once gives bonus points.
Respond with ONLY the action name, nothing else."""

VALID_ACTIONS = ["left", "right", "rotate_cw", "rotate_ccw", "drop", "down", "noop"]

def make_prompt(result):
    """Convert a Tetris result dict into a prompt for the LLM."""
    return f"""Board:
{result['board']}

Current piece: {result['current_piece']} (shape: {result['current_piece_shape']})
Next piece: {result['next_piece']}
Score: {result['score']} | Lines: {result['total_lines']} | Height: {result['max_height']} | Holes: {result['holes']}

Choose your action:"""

def generate_training_prompts(n_prompts=200, max_random_steps=30):
    """Generate diverse board states locally (no network needed)."""
    prompts = []
    for i in range(n_prompts):
        env = TetrisEnv(seed=i)
        result = env.reset(seed=i)
        n_steps = random.randint(0, max_random_steps)
        for _ in range(n_steps):
            action = random.choice(VALID_ACTIONS)
            result = env.step(action)
            if result['done']:
                break
        if not result['done']:
            prompt = make_prompt(result)
            prompts.append({
                "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                           {"role": "user", "content": prompt}],
            })
    print(f"Generated {len(prompts)} training prompts")
    return prompts

train_prompts = generate_training_prompts(200)

In [ ]:
# Cell 4: Generate training prompts from random board states
import random

SYSTEM_PROMPT = """You are a Tetris AI agent. You see a 10x20 board and must choose the best action.
Actions: left, right, rotate_cw, rotate_ccw, drop, down, noop
Your goal: clear as many lines as possible. Clearing multiple lines at once gives bonus points.
Respond with ONLY the action name, nothing else."""

VALID_ACTIONS = ["left", "right", "rotate_cw", "rotate_ccw", "drop", "down", "noop"]

def make_prompt(observation):
    """Convert a Tetris observation into a prompt for the LLM."""
    return f"""Board:
{observation['board']}

Current piece: {observation['current_piece']} (shape: {observation['current_piece_shape']})
Next piece: {observation['next_piece']}
Score: {observation['score']} | Lines: {observation['total_lines']} | Height: {observation['max_height']} | Holes: {observation['holes']}

Choose your action:"""

def generate_training_prompts(n_prompts=200, max_random_steps=30):
    """Generate diverse board states by playing random moves."""
    prompts = []
    client = TetrisClient().connect()

    for i in range(n_prompts):
        result = client.reset(seed=i)
        # Play random steps to get diverse board states
        n_steps = random.randint(0, max_random_steps)
        for _ in range(n_steps):
            action = random.choice(VALID_ACTIONS)
            result = client.step(action)
            if result['done']:
                break

        if not result['done']:
            prompt = make_prompt(result['observation'])
            prompts.append({
                "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                           {"role": "user", "content": prompt}],
            })

    client.close()
    print(f"Generated {len(prompts)} training prompts")
    return prompts

train_prompts = generate_training_prompts(200)

In [ ]:
# Cell 6: Define reward function (LOCAL engine, fast)

def tetris_reward_func(completions, **kwargs):
    """Evaluate each completion by running it in a local Tetris engine."""
    rewards = []
    for i, completion in enumerate(completions):
        text = completion[0]['content'].strip().lower() if isinstance(completion, list) else str(completion).strip().lower()

        # Parse action
        action = None
        for valid in VALID_ACTIONS:
            if valid in text:
                action = valid
                break

        if action is None:
            rewards.append(-10.0)
            continue

        # Run action in local engine
        env = TetrisEnv(seed=i)
        env.reset(seed=i)
        result = env.step(action)
        rewards.append(float(result['reward']))

    return rewards

# Quick test
test_completions = [[{"content": "drop"}], [{"content": "invalid_gibberish"}], [{"content": "left"}]]
test_rewards = tetris_reward_func(test_completions)
print(f"Test rewards: {test_rewards}")

In [ ]:
# Cell 6: Define reward function
import re

def tetris_reward_func(completions, **kwargs):
    """Evaluate each completion by actually playing it in the Tetris environment."""
    rewards = []
    client = TetrisClient().connect()

    for i, completion in enumerate(completions):
        # Extract action from model output
        text = completion[0]['content'].strip().lower() if isinstance(completion, list) else str(completion).strip().lower()

        # Parse action - find first valid action in the text
        action = None
        for valid in VALID_ACTIONS:
            if valid in text:
                action = valid
                break

        if action is None:
            # Invalid action - penalty
            rewards.append(-10.0)
            continue

        # Play the action in a fresh game to evaluate
        try:
            result = client.reset(seed=i)
            result = client.step(action)
            reward = float(result['reward'])
            rewards.append(reward)
        except Exception as e:
            rewards.append(-10.0)

    client.close()
    return rewards

# Quick test
test_completions = [[{"content": "drop"}], [{"content": "invalid_gibberish"}], [{"content": "left"}]]
test_rewards = tetris_reward_func(test_completions)
print(f"Test rewards: {test_rewards}")

In [ ]:
# Cell 7: Configure and run GRPO training
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="tetris-agent-grpo",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,  # N responses per prompt for GRPO
    max_completion_length=16,  # Action is short - just one word
    max_prompt_length=1024,
    learning_rate=5e-6,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[tetris_reward_func],
    args=training_args,
    train_dataset=dataset,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 8: Plot reward curve
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_logs = [l for l in logs if 'loss' in l]
reward_logs = [l for l in logs if 'reward' in l or 'rewards/mean' in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if train_logs:
    axes[0].plot([l.get('step', i) for i, l in enumerate(train_logs)],
                 [l['loss'] for l in train_logs])
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')

reward_key = 'reward' if reward_logs and 'reward' in reward_logs[0] else 'rewards/mean'
if reward_logs:
    axes[1].plot([l.get('step', i) for i, l in enumerate(reward_logs)],
                 [l.get(reward_key, 0) for l in reward_logs])
    axes[1].set_title('Mean Reward (should go up!)')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Reward')

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()
print("Reward curve saved to reward_curve.png")

In [ ]:
# Cell 9: Test trained agent
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

client = TetrisClient().connect()
result = client.reset(seed=123)

total_reward = 0
for step_num in range(50):
    if result['done']:
        break

    prompt_text = make_prompt(result['observation'])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    outputs = model.generate(inputs, max_new_tokens=16, temperature=0.1)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip().lower()

    # Parse action
    action = "noop"
    for valid in VALID_ACTIONS:
        if valid in response:
            action = valid
            break

    result = client.step(action)
    total_reward += result['reward']

    if step_num % 10 == 0:
        print(f"Step {step_num}: action={action}, reward={result['reward']:.1f}, score={result['observation']['score']}")

print(f"\nGame over! Total reward: {total_reward:.1f}, Score: {result['observation']['score']}, Lines: {result['observation']['total_lines']}")
print(f"\nFinal board:")
print(result['observation']['board'])
client.close()

In [ ]:
# Cell 10: Push trained model to HF Hub
model.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
tokenizer.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
print("Model pushed to https://huggingface.co/VortexedSquirrel/tetris-agent-grpo")